In [ ]:
import pandas as pd
from pathlib import Path
import re
import time
import sys

# ==================================================
# FOLDER NAME
# ==================================================

folder = Path("degree_year")

# Output file with ALL original columns + clean columns
output_file = "IPEDS_ALL_ORIGINAL_INFO_CLEAN.csv"

# ==================================================
# AWARD LEVEL MAP
# ==================================================

award_level_map = {
    1: "Certificate < 1 year",
    2: "Certificate 1-2 years",
    3: "Associate",
    4: "Certificate 2-4 years",
    5: "Bachelor",
    6: "Post-baccalaureate certificate",
    7: "Master",
    8: "Post-master certificate",
    9: "Doctor",
    10: "First-professional degree",
    11: "First-professional certificate",
    17: "Doctor - research/scholarship",
    18: "Doctor - professional practice",
    19: "Doctor - other",
}

# ==================================================
# DEGREE GROUP FUNCTION
# ==================================================

def degree_group(x):
    if pd.isna(x):
        return "Unknown"

    try:
        x = int(float(x))
    except:
        return "Unknown"

    if x == 3:
        return "Associate"
    elif x == 5:
        return "Bachelor"
    elif x == 7:
        return "Master"
    elif x in [9, 17, 18, 19]:
        return "Doctor"
    elif x in [1, 2, 4, 6, 8, 10, 11]:
        return "Certificate / Other"
    else:
        return "Unknown"

# ==================================================
# CLEAN TEXT FUNCTION
# ==================================================

def clean_text(value):
    """
    Cleans text safely.
    Prevents real NaN from becoming the string 'nan'.
    """
    if pd.isna(value):
        return ""

    value = str(value).strip()

    if value.lower() in ["nan", "none", "null"]:
        return ""

    value = re.sub(r"\s+", " ", value)

    return value

# ==================================================
# CLEAN NUMBER FUNCTION
# ==================================================

def clean_number(value):
    """
    Converts number safely.
    Missing or bad values become 0.
    """
    if pd.isna(value):
        return 0

    value = str(value).strip()

    if value.lower() in ["nan", "none", "null", ""]:
        return 0

    value = value.replace(",", "")

    return pd.to_numeric(value, errors="coerce")

# ==================================================
# FIND COLUMN SAFELY
# ==================================================

def find_col(df, possible_names):
    """
    Finds the first matching column name from a list.
    Works even if column names have lowercase/uppercase differences.
    """
    col_map = {col.upper(): col for col in df.columns}

    for name in possible_names:
        if name.upper() in col_map:
            return col_map[name.upper()]

    return None

# ==================================================
# GET YEAR FROM FILE NAME
# ==================================================

def get_year_from_filename(filename):
    match = re.search(r"(19|20)\d{2}", filename)
    if match:
        return int(match.group())
    return None

# ==================================================
# PROGRESS BAR
# ==================================================

def show_progress(current, total, start_time, filename=""):
    percent = current / total
    bar_length = 30
    filled_length = int(bar_length * percent)

    bar = "█" * filled_length + "-" * (bar_length - filled_length)

    elapsed = time.time() - start_time

    if current > 0:
        estimated_total = elapsed / current * total
        remaining = estimated_total - elapsed
    else:
        remaining = 0

    sys.stdout.write(
        f"\rLoading: |{bar}| {current}/{total} "
        f"({percent:.1%}) | Elapsed: {elapsed:.1f}s | Left: {remaining:.1f}s | {filename[:35]}"
    )
    sys.stdout.flush()

# ==================================================
# CHECK FOLDER
# ==================================================

if not folder.exists():
    raise FileNotFoundError(f"Folder not found: {folder}")

csv_files = sorted(folder.glob("*.csv"))

if len(csv_files) == 0:
    raise FileNotFoundError(f"No CSV files found inside folder: {folder}")

print(f"Found {len(csv_files)} CSV files.")
print("Starting clean and combine...\n")

# ==================================================
# READ + CLEAN + COMBINE
# ==================================================

all_data = []
start_time = time.time()

for i, file in enumerate(csv_files, start=1):

    show_progress(i, len(csv_files), start_time, file.name)

    try:
        df = pd.read_csv(
            file,
            dtype=str,
            low_memory=False,
            encoding="latin1"
        )

    except Exception as e:
        print(f"\nSkipping file because of error: {file.name}")
        print(e)
        continue

    # ----------------------------------------------
    # Add year from filename
    # ----------------------------------------------

    year = get_year_from_filename(file.name)
    df["clean_year"] = year

    # ----------------------------------------------
    # Find useful IPEDS columns safely
    # ----------------------------------------------

    unitid_col = find_col(df, ["UNITID"])
    cip_col = find_col(df, ["CIPCODE", "CIP", "CIP_CODE"])
    major_col = find_col(df, ["CIPTITLE", "CIPDESC", "MAJOR_NAME", "PROGRAM"])
    awlevel_col = find_col(df, ["AWLEVEL", "AWARD_LEVEL"])
    count_col = find_col(df, ["CTOTALT", "XCTOTALT", "TOTAL", "COUNT"])

    # ----------------------------------------------
    # Clean UNITID
    # ----------------------------------------------

    if unitid_col:
        df["clean_unitid"] = df[unitid_col].apply(clean_text)
    else:
        df["clean_unitid"] = ""

    # ----------------------------------------------
    # Clean CIP code
    # ----------------------------------------------

    if cip_col:
        df["clean_cip_code"] = df[cip_col].apply(clean_text)
    else:
        df["clean_cip_code"] = ""

    # ----------------------------------------------
    # Clean major/program name
    # This prevents "nan" from showing
    # ----------------------------------------------

    if major_col:
        df["clean_major_name"] = df[major_col].apply(clean_text)
    else:
        df["clean_major_name"] = ""

    # ----------------------------------------------
    # Clean award level
    # ----------------------------------------------

    if awlevel_col:
        df["clean_awlevel_code"] = pd.to_numeric(df[awlevel_col], errors="coerce")
        df["clean_award_level"] = df["clean_awlevel_code"].map(award_level_map)
        df["clean_degree_group"] = df["clean_awlevel_code"].apply(degree_group)
    else:
        df["clean_awlevel_code"] = pd.NA
        df["clean_award_level"] = "Unknown"
        df["clean_degree_group"] = "Unknown"

    df["clean_award_level"] = df["clean_award_level"].fillna("Unknown")
    df["clean_degree_group"] = df["clean_degree_group"].fillna("Unknown")

    # ----------------------------------------------
    # Clean degree count
    # ----------------------------------------------

    if count_col:
        df["clean_degree_count"] = df[count_col].apply(clean_number)
    else:
        df["clean_degree_count"] = 0

    df["clean_degree_count"] = pd.to_numeric(
        df["clean_degree_count"],
        errors="coerce"
    ).fillna(0).astype(int)

    # ----------------------------------------------
    # Final cleanup: remove NaN text from object columns
    # ----------------------------------------------

    text_cols = df.select_dtypes(include=["object"]).columns

    for col in text_cols:
        df[col] = df[col].fillna("")
        df[col] = df[col].replace(["nan", "NaN", "None", "NONE", "null", "NULL"], "")

    all_data.append(df)

print("\n\nCombining all files...")

# ==================================================
# COMBINE ALL FILES
# ==================================================

final_dataset = pd.concat(all_data, ignore_index=True)

# ==================================================
# FINAL CLEANUP
# ==================================================

text_cols = final_dataset.select_dtypes(include=["object"]).columns

for col in text_cols:
    final_dataset[col] = final_dataset[col].fillna("")
    final_dataset[col] = final_dataset[col].replace(
        ["nan", "NaN", "None", "NONE", "null", "NULL"],
        ""
    )

# ==================================================
# SAVE FILE
# ==================================================

final_dataset.to_csv(output_file, index=False)

total_time = time.time() - start_time

print("\nDone!")
print(f"Saved file: {output_file}")
print(f"Rows: {len(final_dataset):,}")
print(f"Columns: {len(final_dataset.columns):,}")
print(f"Total time: {total_time:.2f} seconds")

Found 41 CSV files.
Starting clean and combine...

Loading: |███████████████████████-------| 32/41 (78.0%) | Elapsed: 98.6s | Left: 27.7s | 2015_c2015_a_rv.csv